# Scientific Validity of PTD for Mean Estimation

This notebook provides empirical evidence that GLIDE's **Predict-Then-Debias (PTD)** implementation is statistically valid.

**Setup:** We estimate the mean of a binary outcome (e.g., the hallucination rate of an AI system). We have:
- A small set of **true labels** (`y_true`), expensive but unbiased
- A large set of **proxy labels** (`y_proxy`), cheap but potentially biased

PTD combines both to produce confidence intervals that are:
1. **Valid**: they cover the true mean at the specified rate (e.g. 90% confidence)
2. **Shorter**: as compared to those obtained with true labels only, especially when the proxy is strongly correlated with the truth

Unlike CLT-based estimators, PTD constructs its confidence interval from the **empirical distribution of bootstrap estimates**, making it reliable even when $n$ is small or the residual distribution is non-Gaussian. We test these claims empirically across a range of proxy-true correlation levels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glide.confidence_intervals import CLTConfidenceInterval
from glide.core.simulated_datasets import generate_binary_dataset
from glide.estimators import ClassicalMeanEstimator, PTDMeanEstimator

## Experiment Parameters

We fix all parameters up front so every section of this notebook uses a consistent setup. We define:

- `CONFIDENCE_LEVEL`: the confidence level at which we compute confidence intervals.

- `N_TRUE`: the number of human-annotated samples used in each experiment.

- `N_PROXY`: the number of proxy-labeled samples.

- `TRUE_MEAN`: the true mean value of human labels.

- `PROXY_MEAN`: the (biased) proxy mean value.

- `N_SEEDS`: the number of simulations in each Monte Carlo experiment.

> **Note on correlation bounds:** Depending on the values of `TRUE_MEAN` and `PROXY_MEAN`, extreme correlation values (close to -1 or 1) may not be achievable. Correlation sweeps are kept within these limits.

We define the following estimation methods for comparison:

- `True only`: uses true human labels only to compute estimates in the conventional way

- `Proxy only`: uses proxy labels only.

- `PTD`: Predict-Then-Debias with power tuning, details are provided below.

In [ ]:
CONFIDENCE_LEVEL = 0.9  # fixed throughout, 90% Confidence Interval
N_TRUE = 500
N_PROXY = 1000
TRUE_MEAN = 0.55
PROXY_MEAN = 0.5
N_SEEDS = 1000

METHODS = ["True only", "Proxy only", "PTD"]

correlations = np.arange(0.1, 0.95, 0.1)
n_correlations = len(correlations)
correlations_lmh = [
    correlations[n_correlations // 4],
    correlations[n_correlations // 2],
    correlations[3 * n_correlations // 4],
]  # low, medium and high values
corr_labels = ["Low", "Medium", "High"]

## Data Generation

We use `generate_binary_dataset` to simulate a realistic evaluation scenario.
It generates correlated binary labels for `N_TRUE` labeled items and `N_PROXY` unlabeled items. The returned `y_true` array has `np.nan` for unlabeled rows; `y_proxy` is fully populated.

The `correlation` parameter controls the Pearson correlation between true and proxy labels on the labeled subset.

In [ ]:
# Single example dataset for illustration
y_true, y_proxy = generate_binary_dataset(
    n=N_TRUE,
    N=N_PROXY,
    true_mean=TRUE_MEAN,
    proxy_mean=PROXY_MEAN,
    correlation=0.8,
    random_seed=42,
)

In the following sections, we perform Monte Carlo experiments to estimate confidence interval width and coverage.

This consists in running `N_SEEDS` simulations where we generate data, compute a confidence interval, and measure its properties each time. We end up with `N_SEEDS` sample values for each measured quantity that we can use to compute statistics.

The same method is used to evaluate **coverage**: the proportion of times confidence intervals contained the target value.

## Inference Results

We compare three estimation methods:

| Estimation method | Data used | Notes |
|--------|-----------|-------|
| **True only** | `y_true` | Classical CLT confidence interval, the gold standard for validity |
| **Proxy only** | `y_proxy` | Biased, cheap but wrong |
| **PTD** | `y_true` + `y_proxy` (debiased) | Bootstrap percentile interval, valid and efficient |

The function below computes means and standard errors for all three estimation methods.

PTD is implemented by `PTDMeanEstimator`. Its `std` is the standard deviation of the bootstrap distribution, which estimates the standard error of the PTD mean estimate. `ClassicalMeanEstimator` implements conventional mean estimation.

In [ ]:
def generate_estimates(y_true, y_proxy):
    """Return mean and std for all three estimation methods."""
    # --- PTD ---
    ptd_result = PTDMeanEstimator().estimate(
        y_true,
        y_proxy,
        confidence_level=CONFIDENCE_LEVEL,
    )

    # --- Classical baselines ---
    estimator = ClassicalMeanEstimator()
    true_only_result = estimator.estimate(y_true, confidence_level=CONFIDENCE_LEVEL)
    proxy_only_result = estimator.estimate(y_proxy, confidence_level=CONFIDENCE_LEVEL)

    return {
        "True only": {
            "mean": true_only_result.mean,
            "std": true_only_result.std,
        },
        "Proxy only": {
            "mean": proxy_only_result.mean,
            "std": proxy_only_result.std,
        },
        "PTD": {
            "mean": ptd_result.mean,
            "std": ptd_result.std,
            "ess": ptd_result.effective_sample_size,
        },
    }

The three functions below implement the Monte Carlo verification pipeline step by step.

- `monte_carlo_simulation` simulates data for a single correlation level, applies each method, and caches the per-seed outputs.

- `compute_hits` takes the output of `monte_carlo_simulation` and counts how often the target value `TRUE_MEAN` falls inside each method's confidence interval.

- `coverage_with_errbar` estimates the empirical coverage (proportion of hits) along with a confidence interval for that proportion.

In [ ]:
def monte_carlo_simulation(correlation: float, n_seeds=N_SEEDS):
    """Single Monte Carlo loop: cache per-seed mean, std, and ESS for each estimation method."""
    means = {method: np.zeros(n_seeds) for method in METHODS}
    stds = {method: np.zeros(n_seeds) for method in METHODS}
    ess = np.zeros(n_seeds)

    for seed in range(n_seeds):
        y_true, y_proxy = generate_binary_dataset(
            n=N_TRUE,
            N=N_PROXY,
            true_mean=TRUE_MEAN,
            proxy_mean=PROXY_MEAN,
            correlation=correlation,
            random_seed=seed,
        )
        estimates = generate_estimates(y_true, y_proxy)
        for method in METHODS:
            means[method][seed] = estimates[method]["mean"]
            stds[method][seed] = estimates[method]["std"]
        ess[seed] = estimates["PTD"]["ess"]

    stats = {method: {"means": means[method], "stds": stds[method]} for method in METHODS}
    stats["PTD"]["ess"] = ess
    return stats

In [ ]:
def compute_hits(stats, confidence_level):
    """Return per-seed hit indicators {method: float array} at the given confidence level."""
    hits = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=stats[method]["means"], std=stats[method]["stds"], confidence_level=confidence_level
        )
        hits[method] = np.asarray((ci.lower_bound <= TRUE_MEAN) & (TRUE_MEAN <= ci.upper_bound), dtype=float)
    return hits


def coverage_with_errbar(hits, confidence_level):
    """Estimate empirical coverage + Confidence Interval via ClassicalMeanEstimator
    on per-seed hit indicators."""
    estimator = ClassicalMeanEstimator()
    r = estimator.estimate(hits, confidence_level=confidence_level)
    return r.mean, r.confidence_interval.lower_bound, r.confidence_interval.upper_bound

## Coverage Validity

A confidence interval is **valid** if it reliably captures the true value. For example, a 95% confidence interval is valid if, when you repeat the experiment many times, around 95% of those intervals contain the true value.

We check this empirically via a Monte Carlo experiment and count how often each method's confidence interval covers `TRUE_MEAN`.

> **Key question:** Does PTD maintain valid coverage, or does it sacrifice it for shorter intervals?

### Coverage vs confidence level for three correlation levels

We sweep the confidence level from 0.55 to 0.95 and plot the observed coverage.
For a valid estimation method, the dots should fall on or above the black diagonal $y = \text{confidence level}$.

We do this for **low**, **medium**, and **high** proxy correlation.

In [ ]:
# Run simulation for each correlation level
raw_stats = {correlation: monte_carlo_simulation(correlation) for correlation in correlations}

confidence_levels = np.arange(0.55, 1.00, 0.05)

# Derive coverage for every (correlation, confidence_level) pair
coverages_confidence_intervals = {}
for correlation in correlations_lmh:
    coverages_confidence_intervals[correlation] = {}
    for confidence_level in confidence_levels:
        hits = compute_hits(raw_stats[correlation], confidence_level)
        coverages_confidence_intervals[correlation][confidence_level] = dict()
        for method in METHODS:
            coverage_confidence_interval = coverage_with_errbar(hits[method], confidence_level=confidence_level)
            coverages_confidence_intervals[correlation][confidence_level][method] = coverage_confidence_interval

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = {"True only": "steelblue", "PTD": "darkorange", "Proxy only": "red"}

for ax, correlation, label in zip(axes, correlations_lmh, corr_labels):
    ax.plot(confidence_levels, confidence_levels, color="black", lw=1.5, linestyle="--", label="Ideal")
    for method in METHODS:
        mean = [coverages_confidence_intervals[correlation][cl][method][0] for cl in confidence_levels]
        lo = [coverages_confidence_intervals[correlation][cl][method][1] for cl in confidence_levels]
        hi = [coverages_confidence_intervals[correlation][cl][method][2] for cl in confidence_levels]

        ax.plot(confidence_levels, mean, marker="o", color=colors[method], label=method)
        ax.fill_between(confidence_levels, lo, hi, alpha=0.15, color=colors[method])

    ax.set_xlabel("Target confidence level")
    ax.set_ylabel("Observed coverage")
    ax.set_title(f"{label} correlation (${round(correlation, 2)}$)")
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.5, 1.0)

fig.suptitle("Empirical coverage vs target confidence level", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

Both **PTD** and **True only** track the diagonal closely across all correlation levels, confirming that PTD achieves valid coverage regardless of proxy quality. The **Proxy only** method fails because it uses biased data: its coverage of the true mean is close to zero.

---

### Coverage vs correlation for fixed confidence level = 0.9

We now fix the confidence level at 90% and vary the proxy-true correlation from 0.1 to 0.9.
This shows that PTD's validity does not degrade as the proxy becomes weaker.

In [ ]:
coverage_by_corr = {}  # {correlation: {method: observed mean coverage}}
coverage_ci_by_corr = {}  # {correlation: {method: (lower, upper) Confidence Interval on coverage}}

for correlation in correlations:
    hits = compute_hits(raw_stats[correlation], CONFIDENCE_LEVEL)
    coverage_by_corr[correlation] = {}
    coverage_ci_by_corr[correlation] = {}
    for method in METHODS:
        mean_cov, lo, hi = coverage_with_errbar(hits[method], CONFIDENCE_LEVEL)
        coverage_by_corr[correlation][method] = mean_cov
        coverage_ci_by_corr[correlation][method] = (lo, hi)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
method_colors = {"True only": "steelblue", "Proxy only": "red", "PTD": "darkorange"}

for method in ["True only", "PTD"]:
    obs = [coverage_by_corr[correlation][method] for correlation in correlations]
    lo = [coverage_ci_by_corr[correlation][method][0] for correlation in correlations]
    hi = [coverage_ci_by_corr[correlation][method][1] for correlation in correlations]
    ax.plot(correlations, obs, marker="o", color=method_colors[method], label=method)
    ax.fill_between(correlations, lo, hi, alpha=0.15, color=method_colors[method])

ax.axhline(y=CONFIDENCE_LEVEL, color="red", linestyle="--", lw=2, label=f"Target coverage {CONFIDENCE_LEVEL:.0%}")
ax.set_xlabel("Proxy-true correlation")
ax.set_ylabel("Observed coverage")
ax.set_title(f"Coverage vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})")
ax.set_xlim(0, 1)
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

Note that **Proxy only** is not plotted because the proxy is biased (proxy mean differs from true mean). Therefore it has invalid coverage (close to 0) whereas **PTD** and **True only** remain valid across all correlation levels.

---

## Confidence Interval Width

Coverage validity is necessary but not sufficient; we also want **short** intervals. PTD's promise is that by leveraging the unlabeled proxy data, it achieves the same validity as using labeled data alone but with a shorter interval when the proxy is informative.

We report the **mean** and the **10th to 90th percentile band** to capture variability.

In [ ]:
width_by_corr = {}
for correlation in correlations:
    width_by_corr[correlation] = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=raw_stats[correlation][method]["means"],
            std=raw_stats[correlation][method]["stds"],
            confidence_level=CONFIDENCE_LEVEL,
        )
        width_by_corr[correlation][method] = ci.upper_bound - ci.lower_bound

The shaded bands show a percentile range across Monte Carlo runs at the fixed `CONFIDENCE_LEVEL`.
At high correlation, PTD's confidence interval should be substantially narrower than the true-only baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_methods_w = ["True only", "PTD"]
colors_w = {"True only": "steelblue", "PTD": "darkorange"}

# Compute percentiles based on CONFIDENCE_LEVEL
lower_percentile = round(((1 - CONFIDENCE_LEVEL) / 2) * 100)
upper_percentile = 100 - lower_percentile

for method in plot_methods_w:
    means_w = [np.mean(width_by_corr[correlation][method]) for correlation in correlations]
    q_lower = [np.percentile(width_by_corr[correlation][method], lower_percentile) for correlation in correlations]
    q_upper = [np.percentile(width_by_corr[correlation][method], upper_percentile) for correlation in correlations]
    ax.plot(correlations, means_w, marker="o", label=method, color=colors_w[method])
    ax.fill_between(correlations, q_lower, q_upper, alpha=0.15, color=colors_w[method])

ax.set_xlabel("Proxy-true correlation")
ax.set_ylabel("Confidence Interval width")
ax.set_title(
    f"Confidence interval width vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})\n"
    f"Solid = mean, shaded = {lower_percentile:.0f}th to {upper_percentile:.0f}th percentile"
)
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

As expected, PTD's interval width decreases with increasing correlation. Leveraging the unlabeled proxy data is only beneficial when the proxy is informative.

---

## Effective Sample Size

A natural summary of PTD's efficiency gain is the **Effective Sample Size (ESS)**: it is the number of samples needed by the True only estimation method to achieve the same confidence interval width as PTD with the current sample size.

Since confidence interval width $\propto 1/\sqrt{n}$, we can estimate ESS empirically as:

$$\text{ESS} = N_{\text{true}} \times \left(\frac{\bar{w}_{\text{True only}}}{\bar{w}_{\text{PTD}}}\right)^2,$$

where $\bar{w}_{\text{True only}}$ and $\bar{w}_{\text{PTD}}$ are the confidence interval widths for True only and PTD respectively.

When the correlation is zero, ESS $\approx N_{\text{labeled}}$ (no gain). As the correlation approaches $1$, ESS grows. PTD can be equivalent to having a much larger labeled dataset.

In [ ]:
ess_mean = [np.mean(raw_stats[correlation]["PTD"]["ess"]) for correlation in correlations]

ess_q_lower = [np.percentile(raw_stats[correlation]["PTD"]["ess"], lower_percentile) for correlation in correlations]
ess_q_upper = [np.percentile(raw_stats[correlation]["PTD"]["ess"], upper_percentile) for correlation in correlations]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(correlations, ess_mean, marker="o", color="darkorange", label="PTD ESS (mean)")
ax.fill_between(
    correlations,
    ess_q_lower,
    ess_q_upper,
    alpha=0.15,
    color="darkorange",
    label=f"{lower_percentile:.0f}th to {upper_percentile:.0f}th percentile",
)
ax.axhline(y=N_TRUE, color="steelblue", linestyle="--", lw=2, label=f"Baseline (True only, n={N_TRUE})")
ax.set_xlabel("Proxy-true correlation")
ax.set_ylabel("Effective sample size")
ax.set_title("PTD effective sample size vs proxy correlation")
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

---

## Power Tuning

PTD includes a **power tuning** mechanism that estimates the optimal scalar $\hat{\lambda}$ from the bootstrap covariances of labeled means. When the proxy is informative, $\hat{\lambda}$ is large and the estimator borrows heavily from the proxy signal, producing narrower intervals. When the proxy is uninformative, $\hat{\lambda}$ shrinks toward 0 and the estimator falls back to the labeled data only.

Setting `power_tuning=False` fixes $\lambda = 1$, recovering the unweighted PTD estimator. We compare interval widths between the two variants across the full correlation range to verify that power tuning is beneficial or neutral, never harmful.

In [ ]:
METHODS_TUNING = ["True only", "PTD", "PTD (no tuning)"]


def generate_estimates_tuning(y_true, y_proxy):
    """Return mean and std for PTD with and without power tuning."""
    ptd_result = PTDMeanEstimator().estimate(
        y_true,
        y_proxy,
        confidence_level=CONFIDENCE_LEVEL,
        power_tuning=True,
    )
    ptd_no_tuning_result = PTDMeanEstimator().estimate(
        y_true,
        y_proxy,
        confidence_level=CONFIDENCE_LEVEL,
        power_tuning=False,
    )
    estimator = ClassicalMeanEstimator()
    true_only_result = estimator.estimate(y_true, confidence_level=CONFIDENCE_LEVEL)

    return {
        "True only": {"mean": true_only_result.mean, "std": true_only_result.std},
        "PTD": {"mean": ptd_result.mean, "std": ptd_result.std},
        "PTD (no tuning)": {"mean": ptd_no_tuning_result.mean, "std": ptd_no_tuning_result.std},
    }


def monte_carlo_simulation_tuning(correlation: float, n_seeds=N_SEEDS):
    """Monte Carlo loop for power tuning comparison."""
    means = {method: np.zeros(n_seeds) for method in METHODS_TUNING}
    stds = {method: np.zeros(n_seeds) for method in METHODS_TUNING}

    for seed in range(n_seeds):
        y_true, y_proxy = generate_binary_dataset(
            n=N_TRUE,
            N=N_PROXY,
            true_mean=TRUE_MEAN,
            proxy_mean=PROXY_MEAN,
            correlation=correlation,
            random_seed=seed,
        )
        estimates = generate_estimates_tuning(y_true, y_proxy)
        for method in METHODS_TUNING:
            means[method][seed] = estimates[method]["mean"]
            stds[method][seed] = estimates[method]["std"]

    return {method: {"means": means[method], "stds": stds[method]} for method in METHODS_TUNING}


raw_stats_tuning = {correlation: monte_carlo_simulation_tuning(correlation) for correlation in correlations}

In [ ]:
width_by_corr_tuning = {}
for correlation in correlations:
    width_by_corr_tuning[correlation] = {}
    for method in METHODS_TUNING:
        ci = CLTConfidenceInterval(
            mean=raw_stats_tuning[correlation][method]["means"],
            std=raw_stats_tuning[correlation][method]["stds"],
            confidence_level=CONFIDENCE_LEVEL,
        )
        width_by_corr_tuning[correlation][method] = ci.upper_bound - ci.lower_bound

fig, ax = plt.subplots(figsize=(9, 5))
colors_tuning = {"True only": "steelblue", "PTD": "darkorange", "PTD (no tuning)": "forestgreen"}

for method in METHODS_TUNING:
    means_w = [np.mean(width_by_corr_tuning[correlation][method]) for correlation in correlations]
    q_lower = [
        np.percentile(width_by_corr_tuning[correlation][method], lower_percentile) for correlation in correlations
    ]
    q_upper = [
        np.percentile(width_by_corr_tuning[correlation][method], upper_percentile) for correlation in correlations
    ]
    ax.plot(correlations, means_w, marker="o", label=method, color=colors_tuning[method])
    ax.fill_between(correlations, q_lower, q_upper, alpha=0.15, color=colors_tuning[method])

ax.set_xlabel("Proxy-true correlation")
ax.set_ylabel("Confidence Interval width")
ax.set_title(
    f"Effect of power tuning on CI width  (confidence level = {CONFIDENCE_LEVEL:.0%})\n"
    f"Solid = mean, shaded = {lower_percentile:.0f}th to {upper_percentile:.0f}th percentile"
)
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

Power tuning is most effective at high correlation, where it achieves narrower intervals than the fixed $\lambda = 1$ variant. At low correlation the two variants converge, because $\hat{\lambda}$ is estimated to be close to 0 and the PTD estimate relies almost entirely on the labeled data. Both variants remain valid (intervals are never wider than the True only baseline in expectation).

---

## Summary

This notebook has empirically validated that GLIDE's PTD implementation satisfies two key statistical properties:

| Property | Result |
|----------|--------|
| **Coverage validity** | PTD achieves the nominal coverage across all correlation levels and confidence levels tested |
| **Efficiency** | PTD produces shorter confidence intervals than labeled-only whenever correlation is positive, with the gain growing with correlation |

Crucially, the biased baseline (**Proxy only**) fails the coverage test: it appears precise but is systematically wrong. PTD avoids this by correcting for proxy bias using the labeled subset.

The ESS analysis shows that with a proxy correlation of $0.9$, PTD is equivalent to having substantially more labeled data, a significant practical gain in scenarios where true annotation is expensive. The power tuning mechanism further improves efficiency by automatically down-weighting uninformative proxies, ensuring PTD is always at least as efficient as the True only baseline.